# Task 4 eda_original_v2
Refresh the old deterministic image features against the new manifest. Every current image is hash checked.

In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
RAW = SRC / "task4" / "data" / "raw"
import pandas as pd
from src.pipeline.random_seed import set_seed
from src.pipeline.save_data import save_data
SEED = set_seed()
LABELS = {0: "7-11k", 1: "fiveN", 2: "freefried", 3: "pube", 4: "roblock"}
train_manifest = pd.read_csv(RAW / "train.csv")
test_manifest = pd.read_csv(RAW / "test.csv")
assert train_manifest.columns.tolist() == test_manifest.columns.tolist() == ["id", "file_name", "label"]
assert train_manifest.id.is_unique and test_manifest.id.is_unique
assert train_manifest.label.notna().all() and test_manifest.label.isna().all()
assert set(train_manifest.label.astype(int)) == set(LABELS)
assert not (set(train_manifest.id) & set(test_manifest.id))
for manifest, split in ((train_manifest, "train"), (test_manifest, "test")):
    assert manifest.file_name.is_unique
    assert all((RAW / split / f).is_file() for f in manifest.file_name)
print('Train/test rows:', len(train_manifest), len(test_manifest))
print('Classes:', train_manifest.label.value_counts().sort_index().to_dict())

Train/test rows: 31546 25889
Classes: {0: 12429, 1: 4407, 2: 5596, 3: 5926, 4: 3188}


In [2]:
OLD_NAME = "eda_original_v1"
NEW_NAME = "eda_original_v2"
old_card = json.loads((SRC / "task4" / "data" / "eda" / f"{OLD_NAME}.json").read_text(encoding="utf-8"))
old_train = pd.read_parquet(SRC / old_card["splits"]["train"]["path"])
old_test = pd.read_parquet(SRC / old_card["splits"]["test"]["path"])

def rekey(manifest, old_frame, split):
    file_ids = manifest.file_name.str.removesuffix(".jpg")
    assert set(file_ids) == set(old_frame.id)
    selected = old_frame.set_index("id").loc[file_ids].reset_index(drop=True)
    expected = manifest.label.map(LABELS) if split == "train" else manifest.label
    assert selected["class"].reset_index(drop=True).equals(expected.reset_index(drop=True))
    for file_name, stored_hash in zip(manifest.file_name, selected.image_sha256):
        actual_hash = hashlib.sha256((RAW / split / file_name).read_bytes()).hexdigest()
        assert actual_hash == stored_hash, (split, file_name)
    selected["id"] = manifest.id.to_numpy()
    selected["class"] = expected.to_numpy()
    assert selected.id.is_unique
    return selected

train_df = rekey(train_manifest, old_train, "train")
test_df = rekey(test_manifest, old_test, "test")
assert train_df.columns.tolist() == test_df.columns.tolist()
assert not (set(train_df.image_sha256) & set(test_df.image_sha256))
print('Features:', len(test_df.columns)-5)
print('Verified image hashes:', len(train_df)+len(test_df))
info = save_data(task=4, name=NEW_NAME, train=train_df, test=test_df,
    card={"source": "task4/data/raw/train.csv and test.csv with train/*.jpg and test/*.jpg",
          "feature_source": f"{OLD_NAME}.json: unchanged image extraction code and verified SHA-256 for every image",
          "manifest_change": "new id/file_name/label schema; preserve ANS IDs and current test order",
          "leakage_review": "Only per-image features; exact image hashes checked across splits; no fitted preprocessing"})
print(info["splits"]["train"]["path"], info["splits"]["test"]["path"])

Features: 384
Verified image hashes: 57435


task4/data/eda/eda_original_v2-train.parquet task4/data/eda/eda_original_v2-test.parquet
